# Category Handler Output Schema Inspector

Sanity-check tool for the dynamically-built per-category Pydantic output schemas in `schema_factories.py`. Pick any `CategoryName` and print its full JSON schema (the same structure passed to `.chat.completions.parse()` as `response_format`).

Sections:
1. **Category Reference Map** — one-line description, endpoints, and bucket for every category.
2. **Schema Printer** — pick a category, see its full JSON schema.

In [1]:
# Make project root importable when this notebook runs from the
# category_handlers folder rather than the repo root.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
from schemas.enums import CategoryName, EndpointRoute, HandlerBucket
from search_v2.stage_3.category_handlers.schema_factories import (
    OUTPUT_SCHEMAS,
    get_output_schema,
)

## 1. Category Reference Map

Every category, the endpoints it can route to (in priority order), and the handler bucket that determines its output-schema shape. Categories whose endpoint set resolves entirely to non-LLM endpoints (currently only `TRENDING`) have no handler schema — they appear in the map for completeness.

In [2]:
# One-line gloss per category. Concept-level summary — the full
# concept-definition lives on the enum's `description` attribute.
CATEGORY_GLOSS: dict[CategoryName, str] = {
    CategoryName.CREDIT_TITLE:           'Named credits (actor/director/writer/producer/composer) and title-text matches.',
    CategoryName.NAMED_CHARACTER:        'A specific named character appearing in the movie (Batman, Hermione, etc).',
    CategoryName.STUDIO_BRAND:           'Production studio or curated brand (A24, Pixar, Blumhouse, etc).',
    CategoryName.FRANCHISE_LINEAGE:      'Membership and positioning within a named franchise / shared universe.',
    CategoryName.ADAPTATION_SOURCE:      'Source-medium flag (novel/comic/true-story/video-game/etc adaptation).',
    CategoryName.SPECIFIC_SUBJECT:       'A real-world subject or fictional element/motif inside the story.',
    CategoryName.CHARACTER_ARCHETYPE:    'Character type pattern (lovable rogue, femme fatale, anti-hero, etc).',
    CategoryName.AWARDS:                 'Formal award wins/nominations (Oscars, BAFTAs, Cannes, etc).',
    CategoryName.TRENDING:               'Live trending status (no LLM handler — deterministic code path).',
    CategoryName.STRUCTURED_METADATA:    'Closed-schema attribute filters: date, runtime, rating, language, etc.',
    CategoryName.TOP_LEVEL_GENRE:        'Coarse top-level genre (horror/action/comedy/etc).',
    CategoryName.CULTURAL_TRADITION:     'Named cinema tradition (Bollywood, K-cinema, French New Wave, etc).',
    CategoryName.FILMING_LOCATION:       'Where the movie was physically shot.',
    CategoryName.FORMAT_VISUAL:          'Format (doc, anime, mockumentary) + visual-format specifics.',
    CategoryName.SUB_GENRE:              'Sub-genre + story archetype (body horror, cozy mystery, heist, etc).',
    CategoryName.NARRATIVE_DEVICES:      'Storytelling craft (twists, nonlinear, single-location, pacing).',
    CategoryName.TARGET_AUDIENCE:        'Who the movie is packaged for (family, teens, adults, etc).',
    CategoryName.SENSITIVE_CONTENT:      'Sensitive content presence/intensity (violence, nudity, language, etc).',
    CategoryName.SEASONAL_HOLIDAY:       'Seasonal/holiday framing (Christmas, Halloween, summer, etc).',
    CategoryName.PLOT_EVENTS:            'Literal plot events + narrative time/place setting.',
    CategoryName.KIND_OF_STORY:          'Thematic archetype (about grief, redemption, man-vs-nature, etc).',
    CategoryName.VIEWER_EXPERIENCE:      'How the movie feels during viewing (tone, pacing, mood).',
    CategoryName.OCCASION_GOAL:          'Viewing occasion + self-experience goal + comfort-watch framing.',
    CategoryName.CRAFT_ACCLAIM:          'Acclaim attached to a specific craft axis (cinematography, score, etc).',
    CategoryName.RECEPTION_QUALITY:      'Reception judgments + superlatives (cult, acclaimed, best-X-of-Y).',
    CategoryName.POST_VIEWING_RESONANCE: 'How the movie lingers + ending type (haunting, gut-punch, etc).',
    CategoryName.SCALE_SCOPE:            'Scale/scope and holistic vibe queries (epic, intimate, etc).',
    CategoryName.CURATED_CANON:          'Named curated-list membership (Criterion, AFI Top 100, etc).',
    CategoryName.BELOW_THE_LINE:         'Non-indexed creator roles (cinematographer, editor, designer, etc).',
    CategoryName.SOURCE_MATERIAL_AUTHOR: 'Author of the source material being adapted.',
    CategoryName.CHRONOLOGICAL:          'Release-date ordinal position (first/last/newest/oldest).',
    CategoryName.INTERPRETATION_REQUIRED: 'Real meaning that does not map cleanly to any structured category.',
}

# Render the map as a table — one row per category.
rows = []
for cat in CategoryName:
    endpoints = ', '.join(e.value for e in cat.endpoints)
    has_schema = '✓' if cat in OUTPUT_SCHEMAS else '—'
    rows.append((cat.name, cat.value, cat.bucket.value, endpoints, has_schema, CATEGORY_GLOSS[cat]))

name_w   = max(len(r[0]) for r in rows)
label_w  = max(len(r[1]) for r in rows)
bucket_w = max(len(r[2]) for r in rows)
ep_w     = max(len(r[3]) for r in rows)

header = f'{"NAME":<{name_w}}  {"LABEL":<{label_w}}  {"BUCKET":<{bucket_w}}  {"ENDPOINTS":<{ep_w}}  SCHEMA  DESCRIPTION'
print(header)
print('-' * len(header))
for name, label, bucket, endpoints, has_schema, gloss in rows:
    print(f'{name:<{name_w}}  {label:<{label_w}}  {bucket:<{bucket_w}}  {endpoints:<{ep_w}}  {has_schema:^6}  {gloss}')

NAME                     LABEL                                            BUCKET  ENDPOINTS                    SCHEMA  DESCRIPTION
----------------------------------------------------------------------------------------------------------------------------------
CREDIT_TITLE             Credit + title text                              single  entity                         ✓     Named credits (actor/director/writer/producer/composer) and title-text matches.
NAMED_CHARACTER          Named character                                  single  entity                         ✓     A specific named character appearing in the movie (Batman, Hermione, etc).
STUDIO_BRAND             Studio / brand                                   single  studio                         ✓     Production studio or curated brand (A24, Pixar, Blumhouse, etc).
FRANCHISE_LINEAGE        Franchise / universe lineage                     single  franchise_structure            ✓     Membership and positioning within a named 

## 2. Schema Printer

Set `CATEGORY` below to any `CategoryName` member and re-run the cell to see the schema as a field-tree. Output shows just `field_name: type` per line — Optional becomes `| None`, Pydantic sub-models indent into a child block, large enums collapse to `EnumName (N values)`, and Literals over more than a handful of values collapse to `Literal (N values)`. No `$ref`, `$defs`, `additionalProperties`, or full enum dumps.


In [3]:
import enum
import types
from typing import Literal, Union, get_args, get_origin

from pydantic import BaseModel


# Threshold above which Literal / enum value lists collapse to a count.
_INLINE_LIMIT = 6


def _format_type(annotation: object) -> tuple[str, type[BaseModel] | None]:
    """Return (compact type string, nested-model-to-recurse-into).

    The second element is non-None only when the annotation reduces
    to a single Pydantic model worth expanding inline as a child
    block. Lists of models and Optional[Model] both qualify; unions
    of multiple models do not (we just print the union and stop).
    """
    origin = get_origin(annotation)

    # Optional[X] / Union[...]. types.UnionType covers the PEP 604
    # `X | Y` syntax which has a different origin than typing.Union.
    if origin is Union or isinstance(annotation, types.UnionType):
        args = get_args(annotation)
        non_none = tuple(a for a in args if a is not type(None))
        nullable = len(non_none) != len(args)
        if len(non_none) == 1:
            inner_str, inner_model = _format_type(non_none[0])
            return (f'{inner_str} | None' if nullable else inner_str, inner_model)
        parts = [_format_type(a)[0] for a in non_none]
        suffix = ' | None' if nullable else ''
        return (' | '.join(parts) + suffix, None)

    if origin is list:
        (item,) = get_args(annotation)
        item_str, item_model = _format_type(item)
        return (f'list[{item_str}]', item_model)

    if origin is dict:
        k, v = get_args(annotation)
        return (f'dict[{_format_type(k)[0]}, {_format_type(v)[0]}]', None)

    if origin is Literal:
        values = get_args(annotation)
        if len(values) <= _INLINE_LIMIT:
            inner = ', '.join(repr(v) for v in values)
            return (f'Literal[{inner}]', None)
        return (f'Literal ({len(values)} values)', None)

    # Plain class.
    if isinstance(annotation, type):
        if issubclass(annotation, BaseModel):
            return (annotation.__name__, annotation)
        if issubclass(annotation, enum.Enum):
            n = len(list(annotation))
            if n <= _INLINE_LIMIT:
                vals = ', '.join(repr(m.value) for m in annotation)
                return (f'{annotation.__name__}[{vals}]', None)
            return (f'{annotation.__name__} ({n} values)', None)
        return (annotation.__name__, None)

    # Fallback for parameterised generics we did not special-case.
    return (str(annotation), None)


def print_model(
    model: type[BaseModel],
    *,
    indent: str = '',
    is_root: bool = True,
    seen: set[type[BaseModel]] | None = None,
) -> None:
    """Print a Pydantic model as an indented field tree."""
    seen = seen if seen is not None else set()
    if is_root:
        print(model.__name__)
    if model in seen:
        print(f'{indent}└── … (recursive {model.__name__})')
        return
    seen = seen | {model}

    items = list(model.model_fields.items())
    for i, (field_name, field) in enumerate(items):
        is_last = i == len(items) - 1
        branch = '└── ' if is_last else '├── '
        type_str, child_model = _format_type(field.annotation)
        suffix = ''
        if not field.is_required():
            suffix = '  [optional]'
        print(f'{indent}{branch}{field_name}: {type_str}{suffix}')
        if child_model is not None:
            child_indent = indent + ('    ' if is_last else '│   ')
            print_model(child_model, indent=child_indent, is_root=False, seen=seen)


# ── Pick the category to inspect ──────────────────────────────────
CATEGORY = CategoryName.AWARDS

schema_cls = get_output_schema(CATEGORY)
print(f'Category : {CATEGORY.name}  ({CATEGORY.value!r})')
print(f'Bucket   : {CATEGORY.bucket.value}')
print(f'Endpoints: {[e.value for e in CATEGORY.endpoints]}')
print()
print_model(schema_cls)


Category : AWARDS  ('Award records')
Bucket   : single
Endpoints: ['awards']

AwardsOutput
├── requirement_aspects: list[AwardsRequirementAspect]
│   ├── aspect_description: str
│   ├── relation_to_endpoint: str
│   └── coverage_gaps: str | None  [optional]
├── should_run_endpoint: bool
└── endpoint_parameters: AwardEndpointParameters | None  [optional]
    ├── action_role: ActionRole['candidate_identification', 'candidate_reranking']
    ├── parameters: AwardQuerySpec
    │   ├── concept_analysis: str
    │   ├── scoring_shape_label: str
    │   ├── scoring_mode: AwardScoringMode['floor', 'threshold']
    │   ├── scoring_mark: int
    │   ├── ceremonies: list[AwardCeremony (12 values)] | None  [optional]
    │   ├── award_names: list[str] | None  [optional]
    │   ├── category_tags: list[CategoryTag (81 values)] | None  [optional]
    │   ├── outcome: AwardOutcome['winner', 'nominee'] | None  [optional]
    │   └── years: AwardYearFilter | None  [optional]
    │       ├── year_from: 

### Quick sweep — top-level field shape per bucket

Spot-check that every category in a given bucket produces the same top-level field set.


In [4]:
from collections import defaultdict

by_bucket: dict[HandlerBucket, list[tuple[str, list[str]]]] = defaultdict(list)
for cat, schema_cls in OUTPUT_SCHEMAS.items():
    by_bucket[cat.bucket].append((cat.name, list(schema_cls.model_fields.keys())))

for bucket in HandlerBucket:
    entries = by_bucket.get(bucket, [])
    print(f'── {bucket.value.upper()} ({len(entries)} categories) ──')
    for name, fields in entries:
        print(f'  {name:<28}  {fields}')
    print()

── SINGLE (16 categories) ──
  CREDIT_TITLE                  ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  NAMED_CHARACTER               ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  STUDIO_BRAND                  ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  FRANCHISE_LINEAGE             ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  ADAPTATION_SOURCE             ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  AWARDS                        ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  STRUCTURED_METADATA           ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  FILMING_LOCATION              ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  PLOT_EVENTS                   ['requirement_aspects', 'should_run_endpoint', 'endpoint_parameters']
  VIEWER_EXPERIENCE             ['requirement_aspects